## 1. Preparación

Se conecta Google Drive para cargar el vectorizador TF-IDF ya entrenado.
Reutilizo el mismo vectorizador del modelo baseline: las palabras clave de un texto van a ser las palabras con más peso según ese mismo vocabulario,
no un cálculo nuevo desde cero.

In [12]:
from google.colab import drive
drive.mount('/content/drive')

import joblib
import sys
import importlib

sys.path.append('/content/drive/MyDrive/Datasets_TechMind/palabras_clave')
importlib.invalidate_caches()
from limpieza_texto import limpiar_texto

import nltk
nltk.download('stopwords')

RUTA_VECTORIZADOR = '/content/drive/MyDrive/Datasets_TechMind/modelo baseline/vectorizer.pkl'
vectorizador = joblib.load(RUTA_VECTORIZADOR)

print("Vectorizador cargado, vocabulario de tamaño:", len(vectorizador.vocabulary_))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Vectorizador cargado, vocabulario de tamaño: 3000


In [ ]:
import os
print(os.listdir('/content/drive/MyDrive/Datasets_TechMind/palabras_clave'))

## 2. Extraer las palabras clave de un texto

Convierto el texto a su versión TF-IDF (con el mismo vectorizador de
siempre) y me quedo con las palabras que tuvieron mayor peso en ese
vector — esas son las palabras más "características" de ese texto en
particular, según lo que el modelo ya aprendió de todo el dataset.

In [14]:
import numpy as np

def extraer_palabras_clave(texto, cantidad=5):
    texto_limpio = limpiar_texto(texto)
    vector = vectorizador.transform([texto_limpio])

    nombres_palabras = vectorizador.get_feature_names_out()
    pesos = vector.toarray()[0]

    indices_top = pesos.argsort()[::-1][:cantidad]

    palabras_clave = [nombres_palabras[i] for i in indices_top if pesos[i] > 0]
    return palabras_clave

## 3. Probar la función con un texto de ejemplo

Prueba con un texto técnico conocido, aqui revisamos si las palabras que devuelve tienen sentido con el tema real del texto.

In [15]:
texto_prueba = """
Docker es una plataforma que permite empaquetar una aplicación junto con
todas sus dependencias en un contenedor, para que funcione igual sin
importar en qué máquina se ejecute. Kubernetes se usa para orquestar
muchos contenedores Docker en producción.
"""

palabras = extraer_palabras_clave(texto_prueba)
print("Palabras clave:", palabras)

Palabras clave: ['docker', 'funcione', 'dependencias', 'kubernetes', 'contenedor']


## 4. Segunda prueba, con un texto de otro tema

Pruebo con un texto de una categoría distinta, asi confirmamos que la
función se adapta al contenido real y no repite siempre las mismas
palabras.

In [16]:
texto_prueba_2 = """
Python es un lenguaje de programación ampliamente usado en ciencia de
datos. Pandas es una librería que permite trabajar con tablas de datos
de forma sencilla, mientras que Scikit-learn se usa para entrenar
modelos de machine learning.
"""

palabras_2 = extraer_palabras_clave(texto_prueba_2)
print("Palabras clave:", palabras_2)

Palabras clave: ['learn', 'scikit', 'entrenar', 'usado', 'pandas']


## 5. Resumen

La función extrae las 5 palabras con mayor peso TF-IDF de un texto,
reutilizando el mismo vectorizador del modelo baseline — sin entrenar
nada nuevo. Probada con dos textos de temas distintos (Docker/Kubernetes
y Python/Data Science), con resultados coherentes en ambos casos. Nota:
palabras compuestas con guión (como "scikit-learn") pueden aparecer
separadas, porque así quedó definido el vocabulario original del
vectorizador.